In [1]:
import pandas as pd
import numpy as np

In [2]:
bm_raw = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\bm_raw_new.csv")
bss_pca = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\bss_pca_raw.csv")
pca_sellers = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Consumption till 13th\PCA for Sellers.csv")
classification = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\classification_mapping.xlsx")
Brand_mapping = pd.read_excel(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\3.Mar'26\display files\Brand_mapping.xlsx")
search_roin = pd.read_csv(r"G:\.shortcut-targets-by-id\1NZJpN1JTQGJpoUL9k1WHLGD78oymyX1F\Ads Biz-fin\Estimate Working\CY'26\1.Jan'26\14.01\Search\Self serve dashboard.csv")

In [3]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR'],
      dtype='object')

In [4]:
bm_raw = bm_raw[bm_raw['total_cost_x']>5]

In [5]:
# bss_pca.columns

In [6]:
#  pca_sellers.columns

In [7]:
classification.columns

Index(['Super Categories_Tag', 'BU_Tag', 'Wrong Tagging', 'New Tag', 'Type',
       'Adv. Name', 'Tag', 'Remarks', 'Super Categories_HL', 'BU_HL', 'BU_HL2',
       'HL Supercat'],
      dtype='object')

In [8]:
Brand_mapping.columns

Index(['Incorrect Account Name', 'Correct Name', 'BU', 'Concatenate',
       'Advertiser Name', 'Tag', 'Ad Account ID', 'Ad Account Name',
       'Business Account ID', 'Business Account Name', 'Vertical',
       'Current SC', 'New SC'],
      dtype='object')

In [9]:
incorrect_brands = Brand_mapping['Ad Account ID'].unique()

bm_raw['New Alpha/MP'] = np.where(
    bm_raw['advertiser_id'].isin(incorrect_brands) | (bm_raw['Team'] == 'IC'),
    'IC',
    bm_raw['Alpha_MP']
)


In [10]:
bm_raw['New Alpha/MP'].value_counts()

New Alpha/MP
MP       452982
Alpha     42970
IC         2047
Name: count, dtype: int64

In [11]:
bm_raw['FCFF Alpha/MP'] = np.where(bm_raw['marketplace'] == "HYPERLOCAL","HL",bm_raw['Alpha_MP'])

In [12]:
bm_raw['FCFF Alpha/MP'].value_counts()

FCFF Alpha/MP
MP       454825
Alpha     36818
HL         6356
Name: count, dtype: int64

In [13]:
brand_map = dict(zip(Brand_mapping['Vertical'], Brand_mapping['New SC']))
sc_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

conditions = [
    bm_raw['marketplace'] == "Grocery",
    bm_raw['analytic_vertical'].isin(brand_map.keys()),
    bm_raw['analytic_super_category'].isin(brand_map.keys())
]

choices = [
    "Grocery",
    bm_raw['analytic_vertical'].map(brand_map),
    bm_raw['analytic_super_category'].map(brand_map)
]

fallback = bm_raw['analytic_super_category'].map(sc_map).fillna(bm_raw['analytic_super_category'])

bm_raw['New Supercat'] = np.select(conditions, choices, default=fallback)


In [14]:
# print(bm_raw['New Supercat'].value_counts().to_string())


In [15]:
bm_raw['SC match FCFF'] = bm_raw['New Supercat'].isin(classification['Super Categories_Tag'])

In [16]:
bm_raw['SC match FCFF'].value_counts()

SC match FCFF
True     497710
False       289
Name: count, dtype: int64

In [17]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bm_raw['New BU'] = bm_raw['New Supercat'].map(sc_bu_map)

In [18]:
print(bm_raw['New BU'].isnull().sum())

289


In [19]:
bm_raw['Spends'] = bm_raw['total_cost_x']

In [20]:
bm_raw['Spends'].sum()

np.float64(1159516729.8400002)

In [21]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bm_raw['lookup_key'] = bm_raw['advertiser_name'].astype(str) + bm_raw['analytic_super_category'].astype(str)

cond_a = bm_raw['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bm_raw['analytic_super_category'].isin(Brand_mapping['BU'])

bm_raw['Brand'] = np.where(
    cond_a & cond_b,
    bm_raw['advertiser_name'].map(brand_map),
    bm_raw['brand']
)
bm_raw['Brand'] = bm_raw['Brand'].fillna(bm_raw['brand'])


In [22]:
len(bm_raw[bm_raw['Brand'] == "0"])


250

In [23]:
# bm_raw['Brand'].value_counts(normalize=True)

In [24]:
bm_raw.columns

Index(['marketplace', 'campaign_id', 'advertiser_id', 'advertiser_name',
       'BUSINESS_ACCOUNT_ID', 'BUSINESS_ACCOUNT_NAME', 'seller_id',
       'adgroup_id', 'brand', 'commodity_name', 'analytic_vertical',
       'analytic_super_category', 'lifestyle_supercategory', 'Supercategory',
       'BU', 'HL_BU', 'Alpha_MP', 'BMP_Tag', 'Team', 'budget_type',
       'cost_model', 'pacing', 'campaign_type', 'campaign_name', 'status',
       'start_date', 'end_date', 'commodity_id', 'Channel', 'allocated_budget',
       'gross_budget', 'uniqueviewcount', 'actioncount', 'total_cost_x',
       'addtobasketcount', 'viewcount', 'cnt_lid', 'attr_units',
       'attr_revenue', 'overburn_share', 'Actual_spend', 'Budget_adgroup',
       'ROI', 'CTR', 'CVR', 'New Alpha/MP', 'FCFF Alpha/MP', 'New Supercat',
       'SC match FCFF', 'New BU', 'Spends', 'lookup_key', 'Brand'],
      dtype='object')

# BSS PCA

In [25]:
bss_pca.columns

Index(['Campaign ID', 'Marketplace', 'BU', 'Supercategory', 'HL_BU', 'store',
       'analytic_super_category', 'Brands', 'Store Name',
       'creative_template_id', 'creative_type', 'costmodel', 'Ad Account ID',
       'Ad Account Name', 'Business Account ID', 'Business Account Name',
       'Team', 'Alpha_MP', 'BMP_Tag', 'Channel', 'lifestyle_supercategory',
       'spend', 'views', 'clicks', 'ppv', 'click_direct_units',
       'click_indirect_units', 'click_direct_revenue',
       'click_indirect_revenue', 'Product', 'Average CPC', 'CTR', 'ROI'],
      dtype='object')

In [26]:
# bss_pca = bss_pca[bss_pca['spend']>5].copy()
bss_pca = bss_pca.copy()

In [27]:
incorrect_brands = Brand_mapping['Advertiser Name'].unique()

bss_pca['New Alpha/MP'] = np.where(
    bss_pca['Business Account Name'].isin(incorrect_brands) | (bss_pca['Team'] == 'IC'),
    'IC',
    bss_pca['Alpha_MP']
)


In [28]:
bss_pca['New Alpha/MP'].value_counts()

New Alpha/MP
Alpha    215676
MP       108980
Name: count, dtype: int64

In [29]:
# class_map = dict(zip(classification['Wrong Tagging'], classification['New Tag']))

# conditions = [ bss_pca['BU'] == "Grocery", bss_pca['BU'] == "Mobile", bss_pca['Supercategory'] == "PetCare", bss_pca['Supercategory'].isin(class_map.keys()) ]
# choices = [ "Grocery", "Mobile", "Pets", bss_pca['Supercategory'].map(class_map) ]

# bss_pca['New Supercat'] = np.select(conditions, choices, default=bss_pca['Supercategory'])

df_class_unique = classification.drop_duplicates(subset=['Wrong Tagging'])
mapping_dict = dict(zip(df_class_unique['Wrong Tagging'], df_class_unique['New Tag']))

conditions = [
    (bss_pca['BU'] == "Grocery"),
    (bss_pca['BU'] == "Mobile"),
    (bss_pca['Supercategory'] == "PetCare")
]
choices = ["Grocery", "Mobile", "Pets"]
supercat = bss_pca['Supercategory'].map(mapping_dict).fillna(bss_pca['Supercategory'])

bss_pca['New Supercat'] = np.select(conditions, choices, default=supercat)


In [30]:
bss_pca['New Supercat'].isnull().sum()

np.int64(0)

In [31]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [32]:
bss_pca['SC match FCFF'].value_counts()

SC match FCFF
True     318830
False      5826
Name: count, dtype: int64

In [33]:
bss_pca[bss_pca['SC match FCFF'] == False]['New Supercat'].value_counts()

New Supercat
Not Assigned    5826
Name: count, dtype: int64

In [34]:
sc_bu_map = dict(zip(classification['Super Categories_Tag'], classification['BU_Tag']))

bss_pca['New BU'] = bss_pca['New Supercat'].map(sc_bu_map)

In [35]:
print(bss_pca['New BU'].isnull().sum())

5826


In [36]:
brand_map = dict(zip(Brand_mapping['Incorrect Account Name'], Brand_mapping['Correct Name']))

bss_pca['lookup_key'] = bss_pca['Ad Account Name'].astype(str) + bss_pca['Supercategory'].astype(str)

cond_a = bss_pca['lookup_key'].isin(Brand_mapping['Concatenate'])
cond_b = bss_pca['Supercategory'].isin(Brand_mapping['BU'])

bss_pca['Brand'] = np.where(
    cond_a & cond_b,
    bss_pca['Ad Account Name'].map(brand_map),
    bss_pca['Brands']
)
bss_pca['Brand'] = bss_pca['Brand'].fillna(bss_pca['Brands'])

In [37]:
len(bss_pca[bss_pca['Brand'] == "0"])

4699

In [38]:
bss_pca['FCFF Alpha/MP'] = np.where(bss_pca['Alpha_MP'] == "IC","Alpha",np.where(bss_pca['Alpha_MP'] =="3P","MP",bss_pca['Alpha_MP']))

In [39]:
bss_pca['BrandxBU'] = bss_pca['Brand'].astype(str) + bss_pca['BU'].astype(str)
lookup_map = bss_pca.drop_duplicates('BrandxBU').set_index('BrandxBU')['New Supercat'].to_dict()

bss_pca['Check'] = bss_pca['BrandxBU'].map(lookup_map)

In [41]:
bss_pca['New Supercat'] = np.where(bss_pca['SC match FCFF'] == False,np.where(bss_pca['Check'] == "Not Assigned", "Not Assigned",bss_pca['Check']),bss_pca['New Supercat'])

In [42]:
bss_pca['SC match FCFF'] = bss_pca['New Supercat'].isin(classification['Super Categories_Tag'])

In [43]:
print(round(bss_pca[bss_pca['SC match FCFF'] == False ]['spend'].sum(),1))

453.6


# Search ROINS

In [44]:
search_roin.head()

,PURCHASE_ORDER_ID,PURCHASE_ORDER_NAME,AMOUNT,START_DATE,END_DATE,PAYMENT_CYCLE,ACCOUNT_ID,ACCOUNT_NAME,BUSINESS_ACCOUNT_ID,BUSINESS_ACCOUNT_NAME,CREATED_AT,UPDATED_AT
0,00NP1Q1N5F9Z,ECO1020326006801,8000000.0,2026-01-12,NaN,30,YWVXLNIMY3JS,BoultAudio - RO,ZTTKSHCHAAK9,EXOTIC MILE PRIVATE LIMITED,2026-01-12T12:32:42.000Z,2026-01-12T12:32:42.000Z
1,03CNFWSQ6TI1,R266/OT/26/000068/00,90000.0,2026-01-02,2026-01-31,60,TPJDJFPSBNV3,Colgate-Minutes,AXTWDULDJS8Z,Colgate Palmolive India Limited,2026-01-02T06:04:58.000Z,2026-01-02T06:04:58.000Z
2,03V43PQ10X2Y,EmamiAgroHealthy&Tasty/Spices/Jan'26,200000.0,2026-01-01,NaN,30,10N825F1XMU4,Emami Mantra H&T_Grocery PO,8ZHLXE1UH284,Emami Agrotech Ltd,2026-01-01T04:55:01.000Z,2026-01-01T04:55:01.000Z
3,05Y902DHG1JY,AGLPO129645,1000000.0,2026-01-10,NaN,30,WDAPRI0KKX,Eveready_PO,org-WDAPRI0KKX,EVEREADY INDUSTRIES INDIA LTD,2026-01-10T10:04:25.000Z,2026-01-10T10:04:25.000Z
4,0720VUH9VLWT,R167/OT/26/000113/01,105000.0,2026-01-01,2026-01-31,30,TK9W1XK3954O,FKMin_Durex,UKLJEXGRY7HR,Reckitt Benckiser (India) Private Limited,2026-01-12T11:36:45.000Z,2026-01-12T11:36:45.000Z


- Start and End of Current Month

In [45]:
SOM = "2026-01-01"
EOM = "2026-01-31"

In [46]:
search_ro = search_roin[(search_roin['START_DATE'] >= SOM ) & (search_roin['START_DATE'] <= EOM) ].copy()

In [47]:
# search_ro.head()

In [48]:
bm_raw_ro = bm_raw[bm_raw['SC match FCFF'] == True].copy()

In [49]:
bm_raw_ro['total_cost_x'].sum()

np.float64(1159380542.6100001)

In [50]:
bss_pca_ro = bss_pca[bss_pca['SC match FCFF'] == True].copy()

In [51]:
bss_pca_ro['spend'].sum()

np.float64(98617559.47999997)

- Alpha/MP

In [52]:
# alpha_mp_map_bm_raw = dict(zip(bm_raw_ro['advertiser_id'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['FCFF Alpha/MP'].to_dict()
# alpha_mp_map_bss_pca = dict(zip(bss_pca_ro['Ad Account ID'],bm_raw_ro['FCFF Alpha/MP']))
alpha_mp_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['FCFF Alpha/MP'].to_dict()

search_ro['Alpha_MP'] = search_ro['ACCOUNT_ID'].map(alpha_mp_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(alpha_mp_map_bss_pca))

In [53]:
print(round(search_ro.groupby('Alpha_MP')['AMOUNT'].sum()))

Alpha_MP
Alpha    658973236.0
HL        35487747.0
MP       186631911.0
Name: AMOUNT, dtype: float64


- BU

In [54]:
bu_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New BU'].to_dict()
bu_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New BU'].to_dict()

search_ro['New BU'] = search_ro['ACCOUNT_ID'].map(bu_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(bu_map_bss_pca))

In [55]:
search_ro['New BU'].value_counts()

New BU
BGM            341
Grocery        119
Large          115
Lifestyle      104
Electronics     63
Home            22
Furniture        8
Mobile           4
Name: count, dtype: int64

- Brand


In [56]:
brand_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['brand'].to_dict()
brand_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['Brands'].to_dict()

search_ro['New Brand'] = search_ro['ACCOUNT_ID'].map(brand_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(brand_map_bss_pca))

- Super Category

In [57]:
sc_map_bm_raw = bm_raw_ro.drop_duplicates('advertiser_id').set_index('advertiser_id')['New Supercat'].to_dict()
sc_map_bss_pca = bss_pca_ro.drop_duplicates('Ad Account ID').set_index('Ad Account ID')['New Supercat'].to_dict()

search_ro['New Supercategory'] = search_ro['ACCOUNT_ID'].map(sc_map_bm_raw).fillna(search_ro['ACCOUNT_ID'].map(sc_map_bss_pca))

In [59]:
# search_ro['New Supercategory'].value_counts()